[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/paramagnetic_salt/paramagnetic_salt.ipynb)

# Adiabatic Demagnetization

This example computes adiabatic demagnetization, the magnetocaloric effect that cools a paramagnetic salt to millikelvins, in a crystal of the salt run as a magnetic refrigerator between a bath at one end and a load at the other. A paramagnetic salt holds spins that turn freely, each a small magnet with two states, along a field or against it, in a crystal lattice that holds a little heat. A field lines the spins up and takes their entropy away: magnetized in contact with a bath, the salt gives that entropy to the bath as heat, and isolated and then demagnetized, its spins take entropy back from the lattice, and the salt cools. The salt is the magnetic twin of the [rubber band](../rubber_band/). An ideal rubber's internal energy does not depend on its stretch, so its tension comes from its entropy; an ideal paramagnet's internal energy does not depend on its magnetization, so the field it holds its magnetization against comes from its entropy too. Stretching a band at fixed entropy warms it, and so does magnetizing a salt; removing the field at fixed entropy cools the salt. In a crystal, heat takes time to cross the lattice, so a field that changes quickly warms or cools the crystal faster than its reservoirs can take or give the heat, and fronts of temperature run in from the end in contact. Debye and Giauque proposed the method independently, in 1926 and 1927, and in 1933 Giauque and MacDougall cooled a salt of gadolinium sulfate by it to a quarter of a kelvin; Giauque's Nobel Prize in Chemistry of 1949 honoured his work on matter at very low temperatures, as [the National Academy's memoir of him](https://nap.nationalacademies.org/resource/biomems/wgiauque.html) recounts.

The algebra is planar projective spacetime joined with a direction for entropy that has no metric, `Algebra("x+y+t-w0") * Algebra("s0")`, and the state of each cell of the crystal is one element of it: its magnetization, the patch of the plane its spins turn in, `moment * share * magnet_plane.dual()`, and the entropy it holds, `entropies * mv.s.dual()`. Its potentials are one element too, the magnet's field, a plane of space, and the temperature along `mv.s`, and the first law is their pairing with a change of state, `potentials & change`: the field's part is the magnet's work, `field & spin`, the temperature's the heat, and neither part reads the other's. The salt's energy is its lattice's heat alone, so the potentials, the energy's derivative over the state, are the temperature times the plane that reads the lattice's entropy, `temperature * (mv.s - spin_slope)`: the field is the temperature times the entropy the spins would regain per unit of magnetization given up, the twin of the band's tension. The energy's second derivative, the stiffness, a map from a change of state to the change of the potentials, is a sum of two dyads, the potentials with themselves and the magnet's plane with itself; symmetric by its form, it carries the Maxwell relation of the magnetocaloric effect. Inverted at each vertex it is the compliance, which turns a change of field and a change of temperature into the change of state they ask for, and with it each step of the crystal is one solve of its heat balance, the lattice's conduction pulled back through the gradient's adjugate, `gradient.adjugate()(medium(gradient))`.

In the notation of the classical thermodynamics of magnetic systems, the internal energy of a salt of $N$ spins of moment $\mu$ in an applied field $B$ changes as $dU = T\,dS + B\,dM$. With the spins' entropy $S_\mathrm{spin}(M) = Nk\left[\ln 2 - \tfrac{1+m}{2}\ln(1+m) - \tfrac{1-m}{2}\ln(1-m)\right]$, $m = M/N\mu$, and the lattice's heat capacity $C$ constant, the energy is $U = CT$ with $T = T_0\,e^{(S - S_\mathrm{spin})/C}$; then $B = (\partial U/\partial M)_S = -T\,dS_\mathrm{spin}/dM$ gives $M = N\mu\tanh(\mu B/kT)$, Curie's law $M = N\mu^2 B/kT$ at small fields, and the Maxwell relation $(\partial T/\partial M)_S = (\partial B/\partial S)_M = B/C$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack, where
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_states
from examples.util.topology import Topology
from examples.thermodynamics.paramagnetic_salt import render, scenarios

np.set_printoptions(precision=3, suppress=True)

# Planar projective spacetime, its degenerate w weighing a place, and the entropy the salt holds, a direction of its
# own, degenerate, since nothing measures an amount of entropy against a length or a time.
space = Algebra("x+y+")
ga = Algebra("x+y+t-w0") * Algebra("s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# The planes reading a place's run along the crystal and across it.
Upright = ga.gatype.vector().derive.within(space)                             # Vector+
# The magnetic field, a plane of space, and a magnetization, the patch of the plane its spins turn in.
Magnetic = ga.gatype.bivector().derive.within(space)                          # Bivector+
Spin = Magnetic.derive.dual                                                   # Antibivector-
# The plane that reads entropy, the temperature's, and an amount of entropy.
Thermal = ga.gatype.vector().derive.within(Algebra("s0"))                     # Vector+
Amount = Thermal.derive.dual                                                  # Antivector-
# The salt's state, its magnetization and its entropy; its potentials, the field and the temperature; and its
# stiffness, the map from a change of state to the change of its potentials.
State = ga.gatype(Spin + Amount)                                              # Antibivector- + Antivector-
Potential = ga.gatype(Magnetic + Thermal)                                     # Bivector+ + Vector+
Stiffness = ga.gatype((Potential, State))                                     # Potential+ <- State-
# The plane the magnet's field turns the spins in, a unit of magnetization along it, and a unit of entropy; a direction
# along time, and a unit of heat.
magnet_plane = mv.xy                                                          # [] Bivector+
spin = magnet_plane.dual()                                                    # [] Spin-
entropy = mv.s.dual()                                                         # [] Amount-
time = mv.t.dual()                                                            # [] Point-
heat = mv.scalar([1.0]).dual()                                                # [] Heat-

## The salt's energy

The spins hold the entropy of two populations mixed, those along the field and those against it, a function of the magnetization's share of saturation alone: the logarithm of two with no magnetization, nothing at saturation. The lattice holds the rest of the salt's entropy at a constant heat capacity, so its temperature grows exponentially with that entropy, and the salt's energy is the lattice's heat, the heat capacity times the temperature, whatever the magnetization. The potentials are that energy's derivative over the state, written out: the energy grows with the lattice's entropy at the temperature, and the lattice holds the salt's entropy less the spins', so the potentials are the temperature times `mv.s - spin_slope`, the plane that reads the lattice's entropy. Temperatures are in kelvin and fields in tesla. Entropy is in Boltzmann's constant per spin, measured so that the lattice holds none at the bath's temperature, and energy is in kelvin per spin; a magnetization is in kelvin per tesla per spin, so that `field & magnetization` is an energy, its saturation the spin's moment, for an electron's spin a Bohr magneton, 0.672 kelvin per tesla.

In [ ]:
# The spin's moment in kelvin per tesla; the lattice's heat capacity per spin, and the temperature at which it holds no
# entropy, the bath's.
moment, heat_capacity, reference = scenarios.moment, scenarios.heat_capacity, scenarios.reference

In [ ]:
def spin_entropy(shares: Scalar) -> Scalar:
    """The entropy of two-level spins `[...] Scalar+` at these shares of their saturated magnetization `[...] Scalar+`:
    the mixing of their two populations, `(1 + shares) / 2` of them along the field and `(1 - shares) / 2` against
    it."""
    along, against = (1 + shares) / 2, (1 - shares) / 2                       # [...] Scalar+, Scalar+
    return -(along * along.log() + against * against.log())


def potentials(state: State) -> Potential:
    """The salt's potentials `[...] Potential+` in this state `[...] State-`: the field the magnet holds it at, along
    the magnet's plane, and its temperature, along s."""
    shares = (magnet_plane & state) / moment                                  # [...] Scalar+
    # The lattice holds the entropy the spins do not, and its temperature grows exponentially with it.
    lattice_entropy = (mv.s & state) - spin_entropy(shares)                   # [...] Scalar+
    temperature = reference * (lattice_entropy / heat_capacity).exp()         # [...] Scalar+
    # The spins' entropy's slope over the state, a reading of its changes: the spins lose entropy as they line up.
    spin_slope = -(shares.arctanh() / moment) * magnet_plane                  # [...] Bivector+
    return temperature * (mv.s - spin_slope)

## The equation of state

The field the salt holds its magnetization at, over the temperature, depends on the magnetization's share alone, `shares.arctanh() / moment`: the salt is half magnetized by 0.82 tesla at one kelvin, and as far by 8.2 millitesla at ten millikelvins. Inverted, the share is the tanh of the moment times the field over the temperature, the equation of state of spins one half; at small fields it is Curie's law, the magnetization the moment squared times the field over the temperature.

In [ ]:
# The shares of saturation the equation of state is drawn at, from none to nearly saturated, each with the lattice
# holding no entropy.
shares = mv.scalar(np.linspace(0, scenarios.largest_share, scenarios.n_shares)[:, None])   # [shares] Scalar+
magnetizations = moment * shares * spin + spin_entropy(shares) * entropy     # [shares] State-
magnetization_potentials = potentials(magnetizations)                         # [shares] Potential+
render.draw_equation_of_state(magnetizations, magnetization_potentials, moment);

The magnetization rises along Curie's law, the dashed line, while the field is small against the temperature, and bends over toward saturation as it grows, nine tenths saturated at 2.2 tesla per kelvin.

## The magnetocaloric effect

The stiffness is the energy's second derivative, written out as a map over `State`, the space of all changes of state, to the change of the potentials. A change of state warms the lattice by the energy it adds, the potentials paired with the change, over the heat capacity, and the warming scales every potential with the temperature; a change of magnetization turns the spins' entropy's slope, against which the spins' susceptibility at a fixed temperature holds them. Both terms are dyads, and so the stiffness is symmetric: the temperature's rise per magnetization at fixed entropy, `stiffness(spin) & entropy`, is the field's rise per entropy at fixed magnetization, `stiffness(entropy) & spin`, the Maxwell relation, and both are the field over the heat capacity, the magnet's work warming the lattice alone. Over the stiffness along the spin, the field's rise per magnetization at fixed entropy, the first is how far the salt cools per tesla of field removed at fixed entropy. The salt is read where its demagnetization starts, held at the bath's temperature of one kelvin in the magnet's highest field, four tesla, its spins' share of saturation the tanh of the moment times the field over the temperature.

In [ ]:
def stiffness(state: State) -> Stiffness:
    """The salt's stiffness `[...] Potential+ <- State-` in this state `[...] State-`, the energy's second
    derivative."""
    shares = (magnet_plane & state) / moment                                  # [...] Scalar+
    salt_potentials = potentials(state)                                       # [...] Potential+
    temperature = salt_potentials & entropy                                   # [...] Scalar+
    # The lattice warmed by the energy a change adds, over its heat capacity, and the potentials scaled with it.
    warming = salt_potentials * (salt_potentials & State) / (heat_capacity * temperature)   # [...] Potential+ <- State-
    # The spins' entropy's slope turned by a change of magnetization, against the temperature.
    ordering = temperature / (moment ** 2 * (1 - shares ** 2)) * magnet_plane * (magnet_plane & State)   # [...] Potential+ <- State-
    return warming + ordering


def state_at(field: Magnetic, temperature: Scalar) -> State:
    """The state `[...] State-` of the salt held at this field `[...] Bivector+` and temperature `[...] Scalar+`: the
    spins' share the tanh of the moment times the field over the temperature, and the entropy the spins' and the
    lattice's."""
    shares = (moment * (field & spin) / temperature).tanh()                   # [...] Scalar+
    entropies = spin_entropy(shares) + heat_capacity * (temperature / reference).log()   # [...] Scalar+
    return moment * shares * spin + entropies * entropy


# The magnet's highest field, at the bath's temperature, and the salt held there.
highest_field, bath_temperature = scenarios.highest_field, scenarios.bath_temperature
magnetized = state_at(highest_field * magnet_plane, mv.scalar([bath_temperature]))   # [] State-
magnetized_stiffness = stiffness(magnetized)                                  # [] Potential+ <- State-
# The temperature's rise per magnetization at fixed entropy, the field's per entropy at fixed magnetization, and the
# cooling per tesla of field removed at fixed entropy.
temperature_per_magnetization = magnetized_stiffness(spin) & entropy          # [] Scalar+
field_per_entropy = magnetized_stiffness(entropy) & spin                      # [] Scalar+
cooling_per_field = temperature_per_magnetization / (magnetized_stiffness(spin) & spin)   # [] Scalar+

In [ ]:
print("temperature per magnetization at fixed entropy:  ", temperature_per_magnetization.to_array())
print("field per entropy at fixed magnetization:        ", field_per_entropy.to_array())
print("cooling per tesla removed at fixed entropy (K/T):", cooling_per_field.to_array())

Both cross terms are the field over the heat capacity, 133 in these units. The salt cools by 0.20 kelvin per tesla removed, less than the quarter kelvin per tesla, the temperature over the field, of an ideal salt whose lattice held no heat: the lattice's heat, given back to the spins as they disorder, slows the cooling.

## The ideal cycle

The magnet follows the ideal cycle, the one a salt in perfect contact with its reservoirs would run between a bath at one kelvin and a load at ten millikelvins, a rectangle on the temperature–entropy chart. In contact with the bath, the salt is magnetized to four tesla and gives the bath heat; isolated, it is demagnetized at fixed entropy and cools to the load's temperature; in contact with the load, it is demagnetized further, to a millitesla, and takes heat from the load as its spins disorder; isolated again, it is magnetized at fixed entropy and warms back to the bath's temperature. Two corners are given, the highest field at the bath's temperature and the lowest at the load's; at the other two the salt has kept one of their entropies and reached the other reservoir's temperature, its spins holding the entropy the lattice does not hold there, and the potentials give the field. Between the corners the magnet's field runs evenly in its logarithm, so that at a fixed entropy the salt cools evenly in the logarithm of its temperature. The lattice's heat capacity is taken constant, three hundredths of Boltzmann's constant per spin: a real crystal's falls as the cube of its temperature and is far smaller at millikelvins, and what limits a real salt's cooling is its spins' interactions with each other.

In [ ]:
# The load's temperature and the magnet's lowest field, held there; the halvings that find the spins' share.
load_temperature, lowest_field, n_halvings = scenarios.load_temperature, scenarios.lowest_field, scenarios.n_halvings


def shares_holding(spin_entropies: Scalar) -> Scalar:
    """The shares of saturation `[...] Scalar+` at which the spins hold these entropies `[...] Scalar+`, found by
    halving the shares from none to saturation: the spins' entropy falls from the logarithm of two to nothing."""
    lower, upper = 0 * spin_entropies, 0 * spin_entropies + 1                 # [...] Scalar+, Scalar+
    for _ in range(n_halvings):
        middle = (lower + upper) / 2                                          # [...] Scalar+
        # Spins at the middle share holding more entropy than these are short of their share.
        short = spin_entropy(middle) > spin_entropies
        lower, upper = where(short, middle, lower), where(short, upper, middle)
    return (lower + upper) / 2


# The two given corners, the highest field at the bath's temperature and the lowest at the load's, and their entropies.
given_fields = stack((highest_field * magnet_plane, lowest_field * magnet_plane))   # [2] Bivector+
given_temperatures = mv.scalar([[bath_temperature], [load_temperature]])     # [2] Scalar+
given_entropies = mv.s & state_at(given_fields, given_temperatures)          # [2] Scalar+
# Isolated, the salt keeps each entropy, and at the other reservoir's temperature its spins hold what the lattice does
# not: the field that holds them there.
others = mv.scalar([[load_temperature], [bath_temperature]])                 # [2] Scalar+
corner_shares = shares_holding(given_entropies - heat_capacity * (others / reference).log())   # [2] Scalar+
cooled, warmed = (potentials(moment * corner_shares * spin + given_entropies * entropy) & spin) * magnet_plane
corner_fields = stack((warmed, given_fields[0], cooled, given_fields[1]))    # [corners] Bivector+
# The corners on the chart: their entropies and temperatures, around the rectangle; and the lines of equal field at
# the two given corners' fields, across the chart's temperatures.
magnetized_entropy, demagnetized_entropy = given_entropies                   # [] Scalar+, Scalar+
corner_entropies = stack((demagnetized_entropy, magnetized_entropy, magnetized_entropy, demagnetized_entropy))   # [corners] Scalar+
corner_temperatures = mv.scalar([[bath_temperature], [bath_temperature], [load_temperature], [load_temperature]])   # [corners] Scalar+
line_states = state_at(given_fields[:, None], mv.scalar(scenarios.line_temperatures[..., None]))   # [lines, samples] State-
render.draw_cycle(corner_entropies, corner_temperatures, line_states, potentials(line_states));

The ideal cycle runs counter-clockwise: along the top at the bath's temperature, giving it heat as the field rises to four tesla, down at fixed entropy, along the bottom at the load's temperature, taking its heat as the field falls to a millitesla, and up at fixed entropy. The faint lines are the salt's states at those two fields, through the cycle's top left and bottom right corners; the rectangle's area is the work the magnet does on the salt in each cycle.

## The crystal

The crystal is a slab twice as long as it is wide, a grid of triangles whose vertices are its sites, each a point of weight one; its spins all lie along the axis out of the slab, so each turns in the slab's own plane, the magnet's. A vertex's cell, a third of each triangle around it, holds that many spins, one to each unit of area, and lengths are in the slab's width. A field over the crystal, the temperature, varies across each triangle by its corners' ramps: a corner's ramp is the facing side joined with time and entropy, along which the temperature does not vary off the slab, over that join closed by the corner, `(sides & time & entropy) / (sides & time & entropy & corners)`, and its gradient is what it reads across space. The lattice conducts heat down the temperature's gradient, alike along the crystal's two axes, `axes * (Upright & axes)` summed, and the gradient's adjugate gathers the flux into what each cell gives up: the conduction is the conductivity pulled back to the vertices, a form on their temperatures. The bath touches the slab's near end and the load its far end, each through a heat switch.

In [ ]:
# The crystal: its triangles and its vertices, places of weight one; and its lattice's conductivity.
topology, coordinates = scenarios.slab(scenarios.n_along, scenarios.n_across)
vertices = (mv(Upright, coordinates) + mv.w).dual().field()                   # Point-[vertices]
conductivity = scenarios.conductivity
corners = topology.incidence[2, 0] * vertices                                 # [corners] Point-[faces]
facing = topology.off[2, 0] * vertices                                        # [2, corners] Point-[faces]
# Each corner's ramp: the facing side, joined with time and entropy, over that join closed by the corner; its gradient,
# what it reads across space.
sides = facing[0] & facing[1]                                                 # [corners] Antibivector-[faces]
ramps = (sides & time & entropy) / (sides & time & entropy & corners)         # [corners] Plane+[faces]
gradients = ramps.cast(Upright)                                               # [corners] Plane+[faces]
# A triangle's area, half its corners joined along time and entropy, and each vertex's cell, a third of each triangle
# about it.
areas = (corners[0] & corners[1] & corners[2] & time & entropy).norm() / 2    # Scalar+[faces]
cells = ~topology.incidence[2, 0].mean(axis=0) * areas                        # Scalar+[vertices]
# The temperature's gradient across each triangle, and the lattice, conducting alike along the crystal's two axes,
# pulled back to the vertices.
gradient = (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0) * Scalar   # (Plane+ <- Scalar+)[faces <- vertices]
axes = stack((mv.x.dual(), mv.y.dual()))                                      # [axes] Point-
medium = areas * conductivity * (axes * (Upright & axes)).sum(axis=0)         # (Point- <- Plane+)[faces]
conduction = gradient.adjugate()(medium(gradient))                            # (Heat- <- Scalar+)[vertices <- vertices]
# The sites the magnetization is drawn at, the middles of squares of cells.
sites = scenarios.sites(coordinates)                                          # [sites]
render.draw_crystal(topology, vertices, sites, bath_temperature, load_temperature);

The slab's triangles, the bath's bar at its near end and the load's at its far end, both open, and the sites where the magnetization is drawn.

## The step

Each step is implicit in the temperatures. The spins follow the field at once, so a cell's change of state is the compliance, the stiffness inverted at each vertex, applied to the change of the potentials: the field's shortfall from the magnet's along the magnet's plane, and the change of the cell's temperature along `mv.s`. Read for its entropy, the compliance's change per degree at a held field is the heat capacity at a held field over the temperature, and its change per tesla at a held temperature, by the compliance's own Maxwell relation the magnetization's fall per degree at a held field, is the heat a change of field releases. The heat balance of all the cells is then one solve for the change of temperature: what warms them is what the conduction and the links to the reservoirs bring in over the step, less what the change of field takes.

In [ ]:
def step(states: State, field: Magnetic, links: Scalar, reservoirs: Scalar, dt: float) -> State:
    """The crystal's states `State-[vertices]` a step of `dt` on from these, the magnet's field moved to this one `[]
    Bivector+` and each vertex's cell tied by these links' conductances `Scalar+[vertices]` to its reservoir's
    temperature `Scalar+[vertices]`."""
    salt_potentials = potentials(states)                                      # Potential+[vertices]
    temperatures = salt_potentials & entropy                                  # Scalar+[vertices]
    # The compliance, the stiffness inverted vertex by vertex: the change of state per unit of field at a held
    # temperature, and per unit of temperature at a held field.
    compliance = stiffness(states).batch().inverse().field()                  # (State- <- Potential+)[vertices]
    along_field = compliance(magnet_plane)                                    # State-[vertices]
    along_temperature = compliance(mv.s)                                      # State-[vertices]
    # The salt's field's shortfall from the magnet's; the heat capacity at a held field, and the heat a cell takes in
    # to keep its temperature as the field changes, negative as it is magnetized.
    shortfall = (field - salt_potentials) & spin                              # Scalar+[vertices]
    capacities = cells * temperatures * (mv.s & along_temperature)            # Scalar+[vertices]
    taken = cells * temperatures * (mv.s & along_field) * shortfall           # Scalar+[vertices]
    # The heat balance over the step, implicit in the temperatures' change.
    linking = spdiag(links * heat * Scalar)                                   # (Heat- <- Scalar+)[vertices <- vertices]
    balance = spdiag(capacities * heat * Scalar) + dt * (conduction + linking)   # (Heat- <- Scalar+)[vertices <- vertices]
    brought = dt * (linking(reservoirs - temperatures) - conduction(temperatures)) - taken * heat   # Heat-[vertices]
    warming = balance.solve(brought)                                          # Scalar+[vertices]
    return states + shortfall * along_field + warming * along_temperature

## The refrigerator's cycle in the crystal

The crystal starts at the bath's temperature throughout, at the cycle's first corner. In contact with a reservoir the magnet ramps its field fast against the crystal's conduction, so that the end away from the reservoir warms or cools nearly as if isolated while the end in contact holds, and then holds the field for the crystal to settle: four tesla with the bath, a millitesla with the load. The phases in contact take most of the frames, the holds sliced finer at their start, where the crystal changes fastest; the isolated phases, which change the crystal alike everywhere, take few. The animation is a time-lapse at an uneven pace, each frame a few implicit steps. The cell below takes about five seconds, most of it drawing.

In [ ]:
# The run's steps: the corner each step's phase runs from and to and how far between them it lies, each vertex's link
# to its reservoir, each step's time, and the steps of each frame; each vertex's reservoir's temperature; and the
# frames' duration in milliseconds.
from_corners, to_corners, within, conductances, durations, n_steps = scenarios.timeline(coordinates)
links = mv.scalar(conductances[..., None]).field()                            # [steps] Scalar+[vertices]
reservoirs = mv.scalar(scenarios.reservoirs(coordinates)[:, None]).field()    # Scalar+[vertices]
frame_duration_ms = scenarios.frame_duration_ms
# The magnet's field at each step, evenly in its logarithm between the corners its phase runs between.
strengths = (corner_fields & spin).log()                                      # [corners] Scalar+
fields = (strengths[from_corners] * (1 - within) + strengths[to_corners] * within).exp() * magnet_plane   # [steps] Bivector+


def run(states: State) -> Iterator[tuple[State, Potential, Magnetic, Scalar]]:
    """The crystal's states `State-[vertices]` and potentials `Potential+[vertices]`, the magnet's field `[] Bivector+`
    and the links `Scalar+[vertices]` at the start and after each frame's steps, from these states."""
    yield states, potentials(states), fields[0], links[0]
    frame_ends = np.cumsum(n_steps)
    for start, end in zip(frame_ends - n_steps, frame_ends):
        for index in range(start, end):
            states = step(states, fields[index], links[index], reservoirs, durations[index])
        yield states, potentials(states), fields[end - 1], links[end - 1]


# The crystal at the bath's temperature at every vertex, in the field of the cycle's first corner.
warm = mv.scalar(np.full((topology.n_vertices, 1), bath_temperature)).field()   # Scalar+[vertices]
frames = show_states(run(state_at(corner_fields[0], warm)), render.animate_crystal, topology, vertices, sites,
                     bath_temperature, load_temperature, highest_field, moment, name="paramagnetic_salt_crystal",
                     duration_ms=frame_duration_ms)

The slab is tinted by its temperature, evenly in its logarithm from deep blue at a third of a millikelvin through pale yellow to deep red at five kelvin. Around it the magnet's gap fills with the field, as deep as the field is strong, its rim's chevrons running the way the field turns the spins; on it each spin patch's arc runs the way its spins turn, as large as their magnetization. A reservoir's bar touches the slab while it is linked, and under the slab is its temperature along its middle, against the place along it, the dashed lines the bath's and the load's. Magnetized with the bath linked, the crystal warms faster than its heat reaches the bath: its far end runs up to four kelvin while the near end stays close to the bath, and the hold lets the warmth drain to it. Isolated and demagnetized, the crystal cools a hundredfold together, the gap emptying while the magnetization hardly changes. With the load linked and the field lowered to a millitesla, the end at the load stays near it while the rest, which the load's heat has not reached, falls to half a millikelvin, still magnetized; the load's heat then fills the crystal in from its end, the cold front receding and the spins disordering behind it. Isolated again and magnetized, the crystal warms back to the bath's temperature, the cycle closing on its first frame.

In [ ]:
# checks
# Curie's law at small fields: at the smallest magnetization drawn, the magnetization over the field over the
# temperature is the moment squared.
curie = moment * shares[1] / ((magnetization_potentials[1] & spin) / (magnetization_potentials[1] & entropy))
np.testing.assert_allclose(curie.to_array(), moment ** 2, rtol=1e-3)
# The Maxwell relation: the stiffness's two cross terms agree, and are the field over the heat capacity.
np.testing.assert_allclose(temperature_per_magnetization.to_array(), field_per_entropy.to_array(), rtol=1e-9)
np.testing.assert_allclose(temperature_per_magnetization.to_array(), highest_field / heat_capacity, rtol=1e-9)
# The crystal's total entropy and its temperatures at each frame, and the frames that end each phase: the bath's
# stroke and its hold, the demagnetization, the load's stroke and its hold, and the magnetization.
held = np.array([(cells * (mv.s & states)).sites.sum().to_array() for states, *_ in frames])   # [frames]
temperatures = [(salt_potentials & entropy).batch().to_array() for _, salt_potentials, *_ in frames]
_, bath_end, demagnetized_end, _, load_end, _ = np.cumsum(scenarios.n_phase_frames)
# Magnetized with the bath linked, the crystal gives the bath entropy.
assert held[bath_end] < held[0]
# Isolated and demagnetized, it keeps its entropy, and every cell ends colder than a fiftieth of the coldest at the
# start.
np.testing.assert_allclose(held[demagnetized_end], held[bath_end], rtol=1e-2)
assert temperatures[demagnetized_end].max() < temperatures[bath_end].min() / 50
# With the load linked it takes entropy from the load, and settles near the load's temperature throughout.
assert held[load_end] > held[demagnetized_end]
np.testing.assert_allclose(temperatures[load_end], load_temperature, rtol=0.1)
# The cycle closes: the crystal ends with the entropy it started with.
np.testing.assert_allclose(held[-1], held[0], rtol=1e-2)